In [1]:
import cdflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_hk_data(csv_file):
    jmag_hk = pd.read_csv(csv_file)
    jmag_hk.columns = [
        "date",
        "FIB_Bx_low",
        "FIB_By_low",
        "FIB_Bz_low",
        "FOB_Bx_low",
        "FOB_By_low",
        "FOB_Bz_low",
        "FIB_Bx_full",
        "FIB_By_full",
        "FIB_Bz_full",
        "FOB_Bx_full",
        "FOB_By_full",
        "FOB_Bz_full"
    ]

    # Convert the 'date' column to datetime
    jmag_hk['date'] = pd.to_datetime(jmag_hk['date'])

    # Plot all 12 data channels
    fig, axes = plt.subplots(4, 3, figsize=(30, 20))
    channels = jmag_hk.columns[1:]  # Exclude the 'date' column

    for i, ax in enumerate(axes.flatten()):
        ax.scatter(jmag_hk['date'], jmag_hk[channels[i]], label=channels[i], s=1)
        #ax.plot(jmag_hk['date'], jmag_hk[channels[i]], label=channels[i])
        ax.set_title(channels[i])
        ax.set_ylabel('Magnetic Field')
        ax.set_xlabel('Date')
        ax.grid()
        ax.legend()

    plt.tight_layout()
    plt.show()

    return

def get_multiplicative_factors(hk_path, echoed_Bx , echoed_By, echoed_Bz, echoed_epoch):

    jmag_hk = pd.read_csv(hk_path)
    jmag_hk.columns = [
        "date",
        "FIB_Bx_low",
        "FIB_By_low",
        "FIB_Bz_low",
        "FOB_Bx_low",
        "FOB_By_low",
        "FOB_Bz_low",
        "FIB_Bx_full",
        "FIB_By_full",
        "FIB_Bz_full",
        "FOB_Bx_full",
        "FOB_By_full",
        "FOB_Bz_full"
    ]

    # Convert the 'date' column to datetime
    jmag_hk['date'] = pd.to_datetime(jmag_hk['date'])

    # Initialize lists to store multiplicative factors
    factors_Bx = {"FIB_Bx_low": [], "FOB_Bx_low": [], "FIB_Bx_full": [], "FOB_Bx_full": []}
    factors_By = {"FIB_By_low": [], "FOB_By_low": [], "FIB_By_full": [], "FOB_By_full": []}
    factors_Bz = {"FIB_Bz_low": [], "FOB_Bz_low": [], "FIB_Bz_full": [], "FOB_Bz_full": []}

    # Ensure echoed_epoch is timezone-aware to match jmag_hk['date']
    echoed_epoch = [pd.Timestamp(t).tz_localize('UTC') for t in echoed_epoch]

    # Iterate through each time in echoed_epoch
    for i, time in enumerate(echoed_epoch):
        # Find the closest time in jmag_hk['date']
        closest_idx = (jmag_hk['date'] - time).abs().idxmin()
        
        # Calculate multiplicative factors for Bx
        for key in factors_Bx.keys():
            if not pd.isna(jmag_hk.loc[closest_idx, key]) and jmag_hk.loc[closest_idx, key] != 0:
                factors_Bx[key].append(echoed_Bx[i] / jmag_hk.loc[closest_idx, key])
            else:
                factors_Bx[key].append(np.nan)
        
        # Calculate multiplicative factors for By
        for key in factors_By.keys():
            if not pd.isna(jmag_hk.loc[closest_idx, key]) and jmag_hk.loc[closest_idx, key] != 0:
                factors_By[key].append(echoed_By[i] / jmag_hk.loc[closest_idx, key])
            else:
                factors_By[key].append(np.nan)
        
        # Calculate multiplicative factors for Bz
        for key in factors_Bz.keys():
            if not pd.isna(jmag_hk.loc[closest_idx, key]) and jmag_hk.loc[closest_idx, key] != 0:
                factors_Bz[key].append(echoed_Bz[i] / jmag_hk.loc[closest_idx, key])
            else:
                factors_Bz[key].append(np.nan)

    # Calculate and print the average for each list of multiplicative factors
    for key, values in factors_Bx.items():
        avg = np.nanmean(values)
        print(f"Average multiplicative factor for {key}: {avg}")

    for key, values in factors_By.items():
        avg = np.nanmean(values)
        print(f"Average multiplicative factor for {key}: {avg}")

    for key, values in factors_Bz.items():
        avg = np.nanmean(values)
        print(f"Average multiplicative factor for {key}: {avg}")

    return factors_Bx, factors_By, factors_Bz

In [ ]:
# Reading jmag HK data from 2025/03

csv_file = "../DATA/jmag_extracted_hk/JUI_JMAG_2025-03-01_2025-04-01_V001.csv"

plot_hk_data(csv_file)

In [ ]:
# Reading jmag HK from 2024/08

csv_file = "../DATA/jmag_extracted_hk/JUI_JMAG_2024-08-01_2024-09-01_V001.csv"

plot_hk_data(csv_file)

In [ ]:
# Reading jmag HK data from 2024/07

csv_file = "../DATA/jmag_extracted_hk/JUI_JMAG_2024-07-01_2024-08-01_V001.csv"

plot_hk_data(csv_file)

In [ ]:
# Get multiplicative factors for jmag HK data on 2025/03/31

hk_path = "../DATA/jmag_extracted_hk/JUI_JMAG_2025-03-01_2025-04-01_V001.csv"
echoed_path = "../DATA/jmag_echoed/2025/03/31/JUICE_LU_RPWI-PPTD-LWYRPW79710_20250331T030003_V01.cdf"

echoed_cdf = cdflib.CDF(echoed_path)

echoed_epoch = echoed_cdf.varget('Epoch')
echoed_Bx = echoed_cdf.varget('LWT79713')
echoed_By = echoed_cdf.varget('LWT79714')
echoed_Bz = echoed_cdf.varget('LWT79715')

echoed_epoch = cdflib.cdfepoch.to_datetime(echoed_epoch)
echoed_epoch = np.array(echoed_epoch, dtype='datetime64[ms]').astype('O')

# Rotation matrix to go from JMAG frame to JUICE frame
R = np.array([
    [-7.77145961*1e-1,  8.39299198*1e-17,   -6.29320391*1e-1],
    [-9.51729314*1e-17, -1.00000000*1e0,    -1.58371803*1e-17],
    [-6.29320391*1e-1,  4.75864657*1e-17,   7.77145961*1e-1]])

# Rotate the magnetic field vectors to the JUICE frame
Bx_rot = R[0,0]*echoed_Bx + R[0,1]*echoed_By + R[0,2]*echoed_Bz
By_rot = R[1,0]*echoed_Bx + R[1,1]*echoed_By + R[1,2]*echoed_Bz
Bz_rot = R[2,0]*echoed_Bx + R[2,1]*echoed_By + R[2,2]*echoed_Bz
echoed_Bx = Bx_rot
echoed_By = By_rot
echoed_Bz = Bz_rot

factors_Bx, factors_By, factors_Bz = get_multiplicative_factors(hk_path, echoed_Bx, echoed_By, echoed_Bz, echoed_epoch)

In [ ]:
# Get multiplicative factors for jmag HK data on 2024/08/21

hk_path = "../DATA/jmag_extracted_hk/JUI_JMAG_2024-08-01_2024-09-01_V001.csv"
echoed_path = "../DATA/jmag_echoed/2024/08/21/JUICE_LU_RPWI-PPTD-LWYRPW79700_20240821T042902_V01.cdf"

echoed_cdf = cdflib.CDF(echoed_path)

NMD = echoed_cdf.varget('NMD')
epoch = echoed_cdf.varget('Epoch')
epoch = cdflib.cdfepoch.to_datetime(epoch)
epoch = np.array(epoch, dtype='datetime64[ms]').astype('O')

echoed_Bx = extract_double_from_columns(NMD, 9, 17)
echoed_By = extract_double_from_columns(NMD, 17, 25)
echoed_Bz = extract_double_from_columns(NMD, 25, 33)

echoed_Bx = np.where(np.abs(echoed_Bx) < 1e-9, np.nan, echoed_Bx)
echoed_By = np.where(np.abs(echoed_By) < 1e-9, np.nan, echoed_By)
echoed_Bz = np.where(np.abs(echoed_Bz) < 1e-9, np.nan, echoed_Bz)

# Rotation matrix to go from JMAG frame to JUICE frame
R = np.array([
    [-7.77145961*1e-1,  8.39299198*1e-17,   -6.29320391*1e-1],
    [-9.51729314*1e-17, -1.00000000*1e0,    -1.58371803*1e-17],
    [-6.29320391*1e-1,  4.75864657*1e-17,   7.77145961*1e-1]])

# Rotate the magnetic field vectors to the JUICE frame
Bx_rot = R[0,0]*echoed_Bx + R[0,1]*echoed_By + R[0,2]*echoed_Bz
By_rot = R[1,0]*echoed_Bx + R[1,1]*echoed_By + R[1,2]*echoed_Bz
Bz_rot = R[2,0]*echoed_Bx + R[2,1]*echoed_By + R[2,2]*echoed_Bz
echoed_Bx = Bx_rot
echoed_By = By_rot
echoed_Bz = Bz_rot

factors_Bx, factors_By, factors_Bz = get_multiplicative_factors(hk_path, echoed_Bx, echoed_By, echoed_Bz, epoch)

In [ ]:
# Get multiplicative factors for jmag HK data on 2024/07/02

hk_path = "../DATA/jmag_extracted_hk/JUI_JMAG_2024-07-01_2024-08-01_V001.csv"
echoed_path = "../DATA/jmag_echoed/2024/07/02/JUICE_LU_RPWI-PPTD-LWYRPW79700_20240702T003500_V02.cdf"

echoed_cdf = cdflib.CDF(echoed_path)

NMD = echoed_cdf.varget('NMD')
epoch = echoed_cdf.varget('Epoch')
epoch = cdflib.cdfepoch.to_datetime(epoch)
epoch = np.array(epoch, dtype='datetime64[ms]').astype('O')

echoed_Bx = extract_double_from_columns(NMD, 9, 17)
echoed_By = extract_double_from_columns(NMD, 17, 25)
echoed_Bz = extract_double_from_columns(NMD, 25, 33)

echoed_Bx = np.where(np.abs(echoed_Bx) < 1e-9, np.nan, echoed_Bx)
echoed_By = np.where(np.abs(echoed_By) < 1e-9, np.nan, echoed_By)
echoed_Bz = np.where(np.abs(echoed_Bz) < 1e-9, np.nan, echoed_Bz)

# Rotation matrix to go from JMAG frame to JUICE frame
R = np.array([
    [-7.77145961*1e-1,  8.39299198*1e-17,   -6.29320391*1e-1],
    [-9.51729314*1e-17, -1.00000000*1e0,    -1.58371803*1e-17],
    [-6.29320391*1e-1,  4.75864657*1e-17,   7.77145961*1e-1]])

# Rotate the magnetic field vectors to the JUICE frame
Bx_rot = R[0,0]*echoed_Bx + R[0,1]*echoed_By + R[0,2]*echoed_Bz
By_rot = R[1,0]*echoed_Bx + R[1,1]*echoed_By + R[1,2]*echoed_Bz
Bz_rot = R[2,0]*echoed_Bx + R[2,1]*echoed_By + R[2,2]*echoed_Bz
echoed_Bx = Bx_rot
echoed_By = By_rot
echoed_Bz = Bz_rot

factors_Bx, factors_By, factors_Bz = get_multiplicative_factors(hk_path, echoed_Bx, echoed_By, echoed_Bz, epoch)